# Data Model to Data Model Harmonization

First, we import the necessary libraries. `bdikit` is the core library for the BDI Kit, and `BaseStandard` is the base class we'll use to create our custom data standards.

In [1]:
import bdikit as bdi
from bdikit.standards.base import BaseStandard

## Raw Data

Here, we define two raw data models, `raw_data1` and `raw_data2`. These are represented as Python dictionaries. Each dictionary contains attribute names as keys, and the values are dictionaries holding metadata about each attribute, such as its description, possible values, and comments. This represents the source and target data models we want to harmonize.

In [2]:
raw_data1 =  {
    'marital_status': {
        'attribute_description': "A demographic parameter indicating a person's current conjugal status.",
        'value_names': ['Divorced', 'Domestic Partnership', 'Married', 'Never Married', 'Separated', 'Widowed'],
        'value_descriptions': [
            'Indicates a person whose marriage has been legally dissolved and has not remarried.',
            'Indicates a person who is a member of an unmarried couple, including same sex couples, living together in longstanding relationships, that are registered or unregistered.',
            'Indicates a person currently joined in a legally binding matrimonial union. Classify common law marriage as married. Includes married couples living together and not living together.',
            'Indicates a person who has never been married or whose marriages have been annulled.',
            'A person who is separated from their spouse, whether or not there is a legal arrangement.',
            'Indicates a person who is no longer married because of the death of his/her spouse and has not remarried.'
        ]
    },
    'sex': {
        'attribute_description': 'Text designations that identify gender. Gender is described as the assemblage of properties that distinguish people on the basis of their societal roles. [Explanatory Comment 1: Identification of gender is based upon self-report and may come from a form, questionnaire, interview, etc.]',
        'value_names': ['female', 'male', 'unspecified', 'unknown', 'not reported'],
        'value_descriptions': [
            'A person who belongs to the sex that normally produces ova. The term is used to indicate biological sex distinctions, or cultural gender role distinctions, or both.',
            'A person who belongs to the sex that normally produces sperm. The term is used to indicate biological sex distinctions, cultural gender role distinctions, or both.',
            'Not stated explicitly or in detail.',
            'Not known, not observed, not recorded, or refused.',
            'Not provided or available.'
        ]
    },
    'tumor_stage': {
        'attribute_description': 'The stage of the tumor at the time of diagnosis.',
        'value_names': ['1', '2', '3', '4'],
        'value_descriptions': [
            'Stage I: The cancer is small and has not spread.',
            'Stage II: The cancer has grown but has not spread.',
            'Stage III: The cancer is larger and may have spread to nearby tissues or lymph nodes.',
            'Stage IV: The cancer has spread to other parts of the body.'
        ],
        'comment': 'This is a numeric attribute representing tumor stage.'
    }
}


raw_data2 = {
    'patient_gender': {
        'attribute_description': 'The gender of the patient.',
        'value_names': ['0', '1', '2'],
        'value_descriptions': ['', '', ''],
        'comment': '0 = Female, 1 = Male, 2 = Gender is not known.'
    },
    'cancer_stage_TNM': {
        'attribute_description': 'The TNM staging system for cancer.',
        'value_names': ['T1', 'T2', 'T3', 'T4'],
        'value_descriptions': [
            'T1: Tumor size and/or extent of local invasion.',
            'T2: Tumor size and/or extent of local invasion.',
            'T3: Tumor size and/or extent of local invasion.',
            'T4: Tumor size and/or extent of local invasion.'
        ],
        'comment': 'TNM staging system.'
    },
    'relationship_status': {
        'attribute_description': "A social and legal attribute describing an individual's current partnership or marital situation.",
        'value_names': ['Married', 'Single', 'Divorced', 'Widowed'],
        'value_descriptions': [
            'Legally married or in a recognized union.',
            'Never legally married.',
            'Previously married but legally divorced.',
            'Spouse deceased and not remarried.'
        ]
    }
}

To perform schema and value matching, you need to represent your data models as "standards". A standard data model is a Python class that implements a specific interface. BDI-Kit uses this interface to understand the structure and content of your data.

The raw data can be in any format (like the `raw_data1` and `raw_data2`). The crucial part is to create a class that inherits from `bdi.standards.base.BaseStandard` and implements the following methods:

*   `get_attributes()`: This method should return a list of attribute names in your data model.
*   `get_attribute_values(attribute_names)`: This method should return a dictionary where keys are attribute names and values are lists of possible values for that attribute.
*   `get_attribute_metadata(attribute_names)`: This method should return a dictionary with metadata for the requested attributes. The structure of this metadata is important for the matching algorithms.

Below is an example of a `CustomStandard` class that wraps our raw dictionary data.

In [ ]:
class CustomStandard(BaseStandard):
    """
    A custom standard implementation that provides a different set of
    predefined attributes and their metadata. This class is intended for
    demonstration purposes.
    """
    def __init__(self, data):
        self._data = data

    def get_attributes(self):
        """
        Returns a list of all the attributes (strings) of the standard.
        """
        return list(self._data.keys())

    def get_attribute_values(self, attribute_names):
        """
        Returns a dictionary where the keys are attribute names and the values are lists of possible values for each attribute.
        """
        return {attr: self._data[attr]['value_names'] for attr in attribute_names if attr in self._data}

    def get_attribute_metadata(self, attribute_names):
        """
        Returns a dictionary where the keys are attribute names and the values are dictionaries containing metadata for each attribute.
        """
        return {attr: self._data[attr] for attr in attribute_names if attr in self._data}


## Schema and Value Matching

Now, we create instances of our `CustomStandard` class for both the source and target data models. This wraps our raw data dictionaries in the standard interface that the BDI Kit expects.

In [4]:
source = CustomStandard(raw_data1)
target = CustomStandard(raw_data2)

We can now use the methods of our `CustomStandard` instances to inspect the data. Here, we retrieve the metadata for the `tumor_stage` attribute from the source data model.

In [5]:
source.get_attribute_metadata(['tumor_stage'] )

{'tumor_stage': {'attribute_description': 'The stage of the tumor at the time of diagnosis.',
  'value_names': ['1', '2', '3', '4'],
  'value_descriptions': ['Stage I: The cancer is small and has not spread.',
   'Stage II: The cancer has grown but has not spread.',
   'Stage III: The cancer is larger and may have spread to nearby tissues or lymph nodes.',
   'Stage IV: The cancer has spread to other parts of the body.'],
  'comment': 'This is a numeric attribute representing tumor stage.'}}

Similarly, we retrieve the metadata for the `patient_gender` attribute from the target data model.

In [6]:
target.get_attribute_metadata(['patient_gender'] )

{'patient_gender': {'attribute_description': 'The gender of the patient.',
  'value_names': ['0', '1', '2'],
  'value_descriptions': ['', '', ''],
  'comment': '0 = Female, 1 = Male, 2 = Gender is not known.'}}

We use `bdi.match_schema` to find correspondences between the attributes of the source and target data models. The function returns a pandas DataFrame containing the predicted attribute matches and their similarity scores.

In [7]:
attribute_matches = bdi.match_schema(source, target)
attribute_matches.head()

,source_attribute,target_attribute,similarity
0,marital_status,relationship_status,0.902986
1,sex,patient_gender,0.902305
2,tumor_stage,cancer_stage_TNM,0.389646


After schema matching, the next step is value matching. `bdi.match_values` takes the source and target standards, the attribute matches from the previous step, and a matching method (in this case, a large language model, "llm") to find correspondences between the values of the matched attributes. Finally, `bdi.view_value_matches` provides a convenient way to visualize the value matching results.

In [8]:
value_matches = bdi.match_values(source, target, attribute_matches,  method="llm")
bdi.view_value_matches(value_matches)

,source_value,target_value,similarity
0,Divorced,Divorced,1.0
1,Married,Married,1.0
2,Never Married,Single,1.0
3,Widowed,Widowed,1.0
4,Separated,Divorced,0.7
5,Domestic Partnership,Single,0.6


,source_value,target_value,similarity
0,female,0,1.0
1,male,1,1.0
2,unspecified,2,1.0
3,unknown,2,1.0
4,not reported,2,0.9


,source_value,target_value,similarity
0,2,T2,1.0
1,4,T4,1.0
2,1,T1,0.9
3,3,T3,0.9
